# 訂正前後の文から編集 (置換・挿入・削除) を取り出す

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/17_gec_education/edit_extraction.ipynb)

NLP クックブック 17-02 ／ 文法誤り訂正と言語教育

誤り文と訂正文の組から、「どの単語がどの単語に直されたか」という編集の一覧を作りたい。誤り傾向の集計に使える。

In [1]:
!pip install -q fugashi unidic-lite

In [2]:
import difflib
from collections import Counter
from fugashi import Tagger

tagger = Tagger()

def tokenize(text):
    return [(w.surface, w.feature.pos1) for w in tagger(text)]

def extract_edits(src, tgt):
    s, t = tokenize(src), tokenize(tgt)
    sm = difflib.SequenceMatcher(None, [w for w, _ in s], [w for w, _ in t], autojunk=False)
    edits = []
    for op, i1, i2, j1, j2 in sm.get_opcodes():
        if op == "equal":
            continue
        o, c = "".join(w for w, _ in s[i1:i2]), "".join(w for w, _ in t[j1:j2])
        pos = (s[i1:i2] or t[j1:j2])[0][1]                       # 編集箇所の先頭の品詞
        kind = {"replace": "置換", "delete": "余剰", "insert": "脱落"}[op]
        edits.append({"type": f"{kind}:{pos}", "orig": o, "corr": c, "span": (i1, i2)})
    return edits

pairs = [("私は昨日学校を行きました。", "私は昨日学校に行きました。"), ("友達と映画を見に行くました。", "友達と映画を見に行きました。"),
         ("日本の食べ物はおいしいと思うます。", "日本の食べ物はおいしいと思います。"), ("私は本読むのが好きです。", "私は本を読むのが好きです。"),
         ("彼は学生のです。", "彼は学生です。"), ("図書館を勉強します。", "図書館で勉強します。")]
stats = Counter()
for src, tgt in pairs:
    edits = extract_edits(src, tgt)
    stats.update(e["type"] for e in edits)
    print(src, "→", [(e["type"], e["orig"], e["corr"]) for e in edits])
print(stats.most_common())

私は昨日学校を行きました。 → [('置換:助詞', 'を', 'に')]
友達と映画を見に行くました。 → [('置換:動詞', '行く', '行き')]
日本の食べ物はおいしいと思うます。 → [('置換:動詞', '思う', '思い')]
私は本読むのが好きです。 → [('脱落:助詞', '', 'を')]
彼は学生のです。 → [('余剰:助詞', 'の', '')]
図書館を勉強します。 → [('置換:助詞', 'を', 'で')]
[('置換:助詞', 2), ('置換:動詞', 2), ('脱落:助詞', 1), ('余剰:助詞', 1)]


- 「置換:助詞」が多ければ助詞の指導を重点的に、といった分析ができる。
- 語順の入れ替えや、複数の誤りが隣り合う場合は、差分アルゴリズムだけでは人間の直感と違う対応付けになることがある。英語では、言語学的な情報を使って対応付ける ERRANT が標準 (次項)。